In [ ]:
# ============================================================
# BIASCOPE — Notebook 06 (v2): MBIB → Hindi translation
# NLLB-200, sentence-level, repetition controls, quality flags
#
# 1) SAMPLE_MODE=True  -> 200 test rows, v1 vs v2 comparison (quick)
# 2) SAMPLE_MODE=False -> full train/validation/test translation
#
# HINDI LABELS ARE INHERITED FROM ORIGINAL MBIB LABELS.
# Translation never generates or modifies labels.
# ============================================================
!pip -q install -U transformers sentencepiece accelerate

import gc, math, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive

drive.mount("/content/drive")

# ---------------- Config ----------------
SAMPLE_MODE = False             # pehle True, phir False karke full run
SAMPLE_SIZE = 200
MODEL_NAME = "facebook/nllb-200-distilled-1.3B"   # OOM aaye to "facebook/nllb-200-distilled-600M"
SRC_LANG, TGT_LANG = "eng_Latn", "hin_Deva"

BATCH_SIZE = 24
MAX_WORDS_PER_SEGMENT = 30
MAX_NEW_TOKENS = 200
NUM_BEAMS = 4
NO_REPEAT_NGRAM = 4
REPETITION_PENALTY = 1.1
CHUNK_ROWS = 400                   # itni rows ke baad checkpoint save
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- Paths ----------------
BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MULTILINGUAL_DIR = BASE_DIR / "data" / "multilingual"
V1_DIR = MULTILINGUAL_DIR / "mbib_hindi_translation"
V2_DIR = MULTILINGUAL_DIR / "mbib_hindi_translation_v2"
V2_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILES = {s: PROCESSED_DIR / f"mbib_{s}.csv" for s in ["train", "validation", "test"]}
OUTPUT_FILES = {s: V2_DIR / f"mbib_{s}_hi.csv" for s in INPUT_FILES}

print("=" * 70)
print("BIASCOPE — NOTEBOOK 06 (v2) | MBIB → HINDI")
print("=" * 70)
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU (slow!)")
print("Mode:", "SAMPLE" if SAMPLE_MODE else "FULL", "| Model:", MODEL_NAME)

# ---------------- Verify inputs ----------------
for split, path in INPUT_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing MBIB {split} file: {path}")
    d = pd.read_csv(path)
    missing = {"id", "text", "label", "dataset_id"} - set(d.columns)
    if missing:
        raise ValueError(f"{split}: missing columns {missing}")
    print(f"{split:11s} -> {len(d):5d} rows | labels {d['label'].value_counts().to_dict()}")
    del d

# ---------------- Load model ----------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(DEVICE).eval()
TGT_ID = tokenizer.convert_tokens_to_ids(TGT_LANG)
assert TGT_ID != tokenizer.unk_token_id, f"Target language token {TGT_LANG} not found"
print("✓ Model loaded")

# ---------------- Segmentation ----------------
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def segment_text(text, max_words=MAX_WORDS_PER_SEGMENT):
    """Sentence split; lambe segments ko even chunks me todta hai (punctuation-stripped text ke liye)."""
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text or text.lower() == "nan":
        return []
    segs = []
    for s in _SENT_SPLIT.split(text):
        w = s.split()
        if not w:
            continue
        if len(w) <= max_words:
            segs.append(" ".join(w))
        else:
            n_chunks = math.ceil(len(w) / max_words)
            size = math.ceil(len(w) / n_chunks)
            for i in range(0, len(w), size):
                segs.append(" ".join(w[i:i + size]))
    return segs

# ---------------- Translation ----------------
@torch.no_grad()
def _translate_batch(batch):
    try:
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=256).to(DEVICE)
        gen = model.generate(
            **enc,
            forced_bos_token_id=TGT_ID,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            repetition_penalty=REPETITION_PENALTY,
            early_stopping=True,
        )
        return [t.strip() for t in tokenizer.batch_decode(gen, skip_special_tokens=True)]
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(batch) == 1:
            raise
        mid = len(batch) // 2
        return _translate_batch(batch[:mid]) + _translate_batch(batch[mid:])

def translate_segments(segments):
    order = sorted(range(len(segments)), key=lambda i: len(segments[i]))   # length-sorted batching
    out = [None] * len(segments)
    for b in range(0, len(order), BATCH_SIZE):
        idx = order[b:b + BATCH_SIZE]
        dec = _translate_batch([segments[i] for i in idx])
        for i, t in zip(idx, dec):
            out[i] = t
    return out

def translate_texts(texts):
    segs, owners = [], []
    for r, t in enumerate(texts):
        for s in segment_text(t):
            segs.append(s)
            owners.append(r)
    outs = translate_segments(segs) if segs else []
    rows = [[] for _ in texts]
    for r, o in zip(owners, outs):
        rows[r].append(o)
    return [" ".join(x) for x in rows]

# ---------------- Quality flags ----------------
DEV = r"\u0900-\u097F"

def letter_ratios(t):
    dev = len(re.findall(rf"[{DEV}]", str(t)))
    lat = len(re.findall(r"[A-Za-z]", str(t)))
    tot = dev + lat
    return (dev / tot, lat / tot) if tot else (0.0, 0.0)

def uniq_ratio(t):
    w = str(t).split()
    return len(set(w)) / len(w) if w else 1.0

def max_ngram_repeat(t, n=3):
    w = str(t).split()
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return max(Counter(grams).values()) if grams else 0

def quality_flags(src, hi):
    n_en = src.map(lambda t: len(str(t).split()))
    n_hi = hi.map(lambda t: len(str(t).split()))
    lat = hi.map(lambda t: letter_ratios(t)[1])
    ratio = (n_hi + 1) / (n_en + 1)
    f = pd.DataFrame({
        "too_short": (n_hi < 3) & (n_en >= 6),
        "len_ratio": (ratio < 0.4) | (ratio > 2.2),
        "repeat": (n_hi >= 12) & (hi.map(uniq_ratio).lt(0.45) | hi.map(max_ngram_repeat).ge(3)),
        "untranslated": lat > 0.35,
        "bot_url": src.astype(str).str.contains(r"I am a bot|https?://|www\.", case=False, regex=True),
    })
    f["auto_flag"] = f.any(axis=1)
    reason_cols = [c for c in f.columns if c != "auto_flag"]
    f["flag_reasons"] = f[reason_cols].apply(lambda r: ",".join(c for c in reason_cols if r[c]), axis=1)
    return f

# ============================================================
# SAMPLE MODE — v1 vs v2 quick comparison
# ============================================================
if SAMPLE_MODE:
    test = pd.read_csv(INPUT_FILES["test"])
    samp = test.sample(min(SAMPLE_SIZE, len(test)), random_state=SEED).reset_index(drop=True)

    t0 = time.time()
    samp["text_hi_v2"] = translate_texts(samp["text"].tolist())
    sec = time.time() - t0
    print(f"\nTranslated {len(samp)} rows in {sec:.0f}s ({sec / len(samp):.2f}s/row)")
    n_total = sum(len(pd.read_csv(p)) for p in INPUT_FILES.values())
    print(f"Estimated FULL run ({n_total} rows): ~{sec / len(samp) * n_total / 60:.0f} min")

    f2 = quality_flags(samp["text"], samp["text_hi_v2"])
    print(f"\nv2 flag rate: {f2['auto_flag'].mean():.1%} | by reason:",
          {c: int(f2[c].sum()) for c in ["too_short", "len_ratio", "repeat", "untranslated", "bot_url"]})
    print("v2 flag rate by label:", f2.groupby(samp["label"])["auto_flag"].mean().round(3).to_dict())

    v1_path = V1_DIR / "mbib_test_hi.csv"
    if v1_path.exists():
        v1 = pd.read_csv(v1_path, encoding="utf-8-sig")[["id", "text_hi"]].rename(columns={"text_hi": "text_hi_v1"})
        v1["id"] = v1["id"].astype(str)
        samp["id"] = samp["id"].astype(str)
        samp = samp.merge(v1, on="id", how="left")
        f1 = quality_flags(samp["text"], samp["text_hi_v1"])
        print(f"v1 flag rate: {f1['auto_flag'].mean():.1%} | by reason:",
              {c: int(f1[c].sum()) for c in ["too_short", "len_ratio", "repeat", "untranslated", "bot_url"]})

    pd.set_option("display.max_colwidth", 160)
    cols = ["id", "label", "text", "text_hi_v1", "text_hi_v2"] if "text_hi_v1" in samp else ["id", "label", "text", "text_hi_v2"]
    print("\nSide-by-side (10 rows):")
    display(samp[cols].head(10))

    out = V2_DIR / "sample_comparison.csv"
    samp.to_csv(out, index=False, encoding="utf-8-sig")
    print("\nSaved:", out)
    print("\nNEXT: sample theek lage to SAMPLE_MODE = False karke cell dobara chalao (full run).")

# ============================================================
# FULL MODE — translate all splits (resume supported)
# ============================================================
else:
    def translate_split(split, in_path, out_path):
        print("\n" + "=" * 70 + f"\nTRANSLATING: {split.upper()}\n" + "=" * 70)
        df = pd.read_csv(in_path)
        df["text_hi"] = pd.Series([None] * len(df), dtype="object")

        if out_path.exists():
            ex = pd.read_csv(out_path, encoding="utf-8-sig")
            if (len(ex) == len(df) and "text_hi" in ex.columns
                    and (ex["id"].astype(str).values == df["id"].astype(str).values).all()):
                df["text_hi"] = ex["text_hi"].astype("object")
                print("Resuming from existing file.")

        def pending_mask():
            return df["text_hi"].isna() | df["text_hi"].astype(str).str.strip().eq("")

        pending = df.index[pending_mask()].tolist()
        print("Rows requiring translation:", len(pending))
        t0 = time.time()

        for c in range(0, len(pending), CHUNK_ROWS):
            idx = pending[c:c + CHUNK_ROWS]
            df.loc[idx, "text_hi"] = translate_texts(df.loc[idx, "text"].tolist())
            df.to_csv(out_path, index=False, encoding="utf-8-sig")
            done = min(c + CHUNK_ROWS, len(pending))
            print(f"  [{done}/{len(pending)}] elapsed={(time.time() - t0) / 60:.1f} min")

        original = pd.read_csv(in_path)
        assert (original["label"].values == df["label"].values).all(), "LABEL MISMATCH"
        assert (original["id"].astype(str).values == df["id"].astype(str).values).all(), "ID MISMATCH"
        df["translation_model"] = MODEL_NAME
        df.to_csv(out_path, index=False, encoding="utf-8-sig")

        print("Missing:", int(df["text_hi"].isna().sum()),
              "| Empty:", int(df["text_hi"].astype(str).str.strip().eq("").sum()))
        print("✓ Labels and IDs preserved | Saved:", out_path)
        return df

    translated = {s: translate_split(s, INPUT_FILES[s], OUTPUT_FILES[s]) for s in INPUT_FILES}

    # ---- quality flags + filtered files (v1 comparison) ----
    print("\n" + "=" * 70 + "\nQUALITY FLAGS (v2 vs v1)\n" + "=" * 70)
    for s, df in translated.items():
        f = quality_flags(df["text"], df["text_hi"])
        out = pd.concat([df, f], axis=1)
        keep = out[~out["auto_flag"]].drop(columns=["too_short", "len_ratio", "repeat",
                                                    "untranslated", "bot_url", "auto_flag", "flag_reasons"])
        keep.to_csv(V2_DIR / f"mbib_{s}_hi_filtered.csv", index=False, encoding="utf-8-sig")
        msg = f"{s:11s} v2 flagged {f['auto_flag'].mean():.1%}"
        v1p = V1_DIR / f"mbib_{s}_hi.csv"
        if v1p.exists():
            v1 = pd.read_csv(v1p, encoding="utf-8-sig")
            msg += f" | v1 flagged {quality_flags(v1['text'], v1['text_hi'])['auto_flag'].mean():.1%}"
        print(msg, "| kept", len(keep), "| labels", keep["label"].value_counts().to_dict())

    print("\nSample (train):")
    pd.set_option("display.max_colwidth", 200)
    display(translated["train"][["id", "label", "text", "text_hi"]].sample(10, random_state=SEED))

    print("\n✅ FULL TRANSLATION DONE — outputs in:", V2_DIR)
    print("NEXT: translation audit cell (path = mbib_hindi_translation_v2) + 100-row human check.")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# BIASCOPE — Hindi news gold-set sampler (FIXED)
# Sources (non-commercial research licenses, cite in report):
#   1) ai4bharat/IndicHeadlineGeneration (hi)
#   2) csebuetnlp/xlsum (hindi)   <- fallback
# Direct parquet loading: dataset-script errors se bachta hai
# ============================================================
import re
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
from huggingface_hub import HfApi, hf_hub_download

drive.mount("/content/drive")

SEED = 42
N_LEX, N_RANDOM = 150, 150
OUT_DIR = Path("/content/drive/MyDrive/BIASCOPE/data/multilingual/gold")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCES = [
    dict(repo="ai4bharat/IndicHeadlineGeneration", config="hi", head_col="target", body_col="input"),
    dict(repo="csebuetnlp/xlsum", config="hindi", head_col="title", body_col="text"),
]
SPLITS = ["test", "validation"]
REVISIONS = [None, "refs/convert/parquet"]
api = HfApi()

def find_files(repo, revision):
    try:
        return api.list_repo_files(repo, repo_type="dataset", revision=revision)
    except Exception as e:
        print(f"  list_repo_files failed ({repo}, rev={revision}): {repr(e)[:200]}")
        return []

def pick_files(files, config, split):
    out = []
    for f in files:
        fl = f.lower()
        if fl.endswith(".parquet") and fl.split("/")[0] == config and split in fl:
            out.append(f)
    return sorted(out)

# ---------------- load ----------------
df, used = None, None
for src in SOURCES:
    for rev in REVISIONS:
        files = find_files(src["repo"], rev)
        for split in SPLITS:
            picks = pick_files(files, src["config"], split)
            print(f"{src['repo']} | rev={rev} | split={split} -> {len(picks)} parquet file(s)")
            if not picks:
                continue
            try:
                parts = [pd.read_parquet(hf_hub_download(src["repo"], f, repo_type="dataset", revision=rev))
                         for f in picks[:3]]
                tmp = pd.concat(parts, ignore_index=True)
            except Exception as e:
                print("  read failed:", repr(e)[:200])
                continue
            if {src["head_col"], src["body_col"]} <= set(tmp.columns):
                df, used = tmp, {**src, "revision": rev, "split": split}
                break
            print("  columns mismatch:", tmp.columns.tolist())
        if df is not None:
            break
    if df is not None:
        break

if df is None:
    print("\nKoi source load nahi hua. Diagnostic (pehle source ki parquet files):")
    for rev in REVISIONS:
        fl = [f for f in find_files(SOURCES[0]["repo"], rev) if f.endswith(".parquet")]
        print(f"rev={rev}: {len(fl)} files, first 15:", fl[:15])
    raise RuntimeError("Upar ka poora output mujhe bhej do, main loader fix kar dunga.")

print("\nLoaded:", used)
print("Rows:", len(df), "| columns:", df.columns.tolist())

# ---------------- candidate sentences ----------------
def norm(t):
    return re.sub(r"\s+", " ", str(t)).strip()

def first_sentence(body):
    parts = re.split(r"(?<=[।?!])\s+", norm(body))
    return parts[0] if parts else ""

def n_words(t):
    return len(str(t).split())

heads = pd.DataFrame({"text_hi": df[used["head_col"]].map(norm), "source_type": "headline"})
leads = pd.DataFrame({"text_hi": df[used["body_col"]].map(first_sentence), "source_type": "lead_sentence"})
heads = heads[heads["text_hi"].map(n_words).between(6, 30)]
leads = leads[leads["text_hi"].map(n_words).between(8, 45)]

cand = pd.concat([heads, leads], ignore_index=True).drop_duplicates("text_hi")
cand = cand[cand["text_hi"].str.contains(r"[\u0900-\u097F]")]          # Devanagari wale hi
cand = cand[cand["text_hi"].str.count(r"[A-Za-z]") < 0.3 * cand["text_hi"].str.len()]  # Latin-heavy hatao
print("Candidates:", len(cand), cand["source_type"].value_counts().to_dict())
assert len(cand) >= N_LEX + N_RANDOM, "Candidates kam hain — SPLITS/filters check karo"

# ---------------- emotive / sensational lexicon (enrichment) ----------------
LEX = ["शर्मनाक", "हैरान", "चौंक", "सनसनी", "खुलासा", "तबाही", "तबाह", "झटका", "बवाल", "हंगामा",
       "खौफ", "दहशत", "कोहराम", "साजिश", "गद्दार", "देशद्रोही", "बेशर्म", "करारा जवाब", "मुंहतोड़",
       "पोल खोल", "भड़क", "बौखला", "आग बबूला", "धज्जियां", "लताड़", "फटकार", "घोटाला", "निकम्मा",
       "नाकाम", "बर्बाद", "ज़हर", "जहर उगल", "चुनौती दी", "ललकार"]
cand["lex_hit"] = cand["text_hi"].map(lambda t: any(w in t for w in LEX))
print(f"Lexicon-hit rate in candidates (base rate): {cand['lex_hit'].mean():.1%}")

lex = cand[cand["lex_hit"]]
rnd = cand[~cand["lex_hit"]]
n_lex = min(N_LEX, len(lex))
n_rnd = min(N_RANDOM + (N_LEX - n_lex), len(rnd))     # lexicon hits kam hon to random se bharo
gold = pd.concat([
    lex.sample(n_lex, random_state=SEED),
    rnd.sample(n_rnd, random_state=SEED),
]).sample(frac=1, random_state=SEED).reset_index(drop=True)

gold.insert(0, "gold_id", [f"HG-{i + 1:03d}" for i in range(len(gold))])
for c in ["bias_label", "subjectivity", "sensationalism", "loaded_language",
          "political_framing", "emotional_tone", "biased_span", "usable", "confidence", "notes"]:
    gold[c] = ""
gold["source_dataset"] = f"{used['repo']} ({used['config']}, {used['split']})"

out = OUT_DIR / "hindi_gold_candidates.csv"
gold.to_csv(out, index=False, encoding="utf-8-sig")
print("\nSaved:", out, "| rows:", len(gold))
print("Composition (lexicon hit):", gold["lex_hit"].value_counts().to_dict(),
      "| type:", gold["source_type"].value_counts().to_dict())
print("\nPreview:")
for t in gold["text_hi"].head(8):
    print("-", t)